# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [72]:
# Install required packages
!pip -q install datasets pandas pyarrow

In [73]:
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN").strip()

# Tables used in this assignment
tables = {
    "clients": load_dataset(
        "FlyRank/internship-warehouse",
        "dim_clients",
        streaming=True,
        token=HF_TOKEN
    )["train"],

    "content": load_dataset(
        "FlyRank/internship-warehouse",
        "dim_content",
        streaming=True,
        token=HF_TOKEN
    )["train"],

    "performance": load_dataset(
        "FlyRank/internship-warehouse",
        "fact_content_daily_performance",
        streaming=True,
        token=HF_TOKEN
    )["train"]
}

print("Datasets loaded successfully!")
print(tables.keys())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Datasets loaded successfully!
dict_keys(['clients', 'content', 'performance'])


In [74]:
# Preview the datasets

for name, ds in tables.items():
    print("=" * 80)
    print(name.upper())
    first = next(iter(ds))
    print(pd.Series(first))

CLIENTS
client_hash_id         client_04660893ae39614a
is_active                                 True
has_gsc_access                            True
has_ga4_access                            True
access_profile                     gsc_and_ga4
client_created_date                 2026-04-15
client_updated_date                 2026-06-27
gsc_data_start                            None
ga4_data_start                      2026-05-22
dtype: object
CONTENT
client_hash_id                 client_04660893ae39614a
content_hash_id               content_004de9653278b5a4
keyword_hash_id               keyword_e754999ab88dd9f2
url_hash_id                       url_d6091f18cf628794
keyword_char_count                                  22
keyword_token_count                                  4
url_char_count                                     108
content_created_date                        2026-05-30
content_updated_date                        2026-07-01
content_type                           keyword artic

In [75]:
for name, ds in tables.items():
    print("=" * 80)
    print(name)

    df = pd.DataFrame(list(ds.take(5)))

    print(df.columns.tolist())

clients
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
content
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
performance
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions

In [76]:
for name, ds in tables.items():
    first = next(iter(ds))
    print("="*60)
    print(name)
    print(list(first.keys()))

clients
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
content
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
performance
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions

In [77]:
for name, ds in tables.items():
    first = next(iter(ds))

    print("\n" + "="*70)
    print(name.upper())
    print("="*70)

    for col in first.keys():
        print(col)


CLIENTS
client_hash_id
is_active
has_gsc_access
has_ga4_access
access_profile
client_created_date
client_updated_date
gsc_data_start
ga4_data_start

CONTENT
client_hash_id
content_hash_id
keyword_hash_id
url_hash_id
keyword_char_count
keyword_token_count
url_char_count
content_created_date
content_updated_date
content_type
search_volume
competition
competition_level
cpc
main_intent
backlinks
category_count
keyword_created_date
provider_used
model_used
char_count
word_count
last_optimized_date
optimization_eligible_date
is_published
is_deleted

PERFORMANCE
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Rule

Recommend content that has:

- High search volume
- Low GSC clicks
- Poor average search position

Reason Codes:

- HIGH_SEARCH_VOLUME
- LOW_CLICKS
- LOW_RANKING


In [78]:
# Signal 1: does CTR drop as position gets worse? (behind the CTR-fix logic)
sample = pd.DataFrame(list(tables["performance"].take(20000)))
sample = sample[sample["gsc_data_available"] == True]   # only rows where GSC data is real

sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"].replace(0, pd.NA)

bins = [0, 3, 10, 20, 1000]
labels = ["1-3", "4-10", "11-20", "21+"]
sample["position_bucket"] = pd.cut(sample["gsc_avg_position"], bins=bins, labels=labels)

bucket_table = sample.groupby("position_bucket").agg(
    n=("ctr", "count"),
    avg_ctr=("ctr", "mean")
)
display(bucket_table)

/tmp/ipykernel_1829/3536831834.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table = sample.groupby("position_bucket").agg(


,n,avg_ctr
position_bucket,,
1-3,399,0.034434
4-10,5474,0.013630
11-20,3682,0.009151
21+,10400,0.002986


In [79]:
# Signal 2: does high search_volume + low clicks indicate a "quick win"? (behind volume flag)
content_full = pd.DataFrame(list(tables["content"]))   # full table, same as Cell 14
merged = sample.merge(content_full, on=["client_hash_id", "content_hash_id"], how="inner")

vol_bins = [0, 10, 50, 200, 100000]
vol_labels = ["low", "medium", "high", "very_high"]
merged["volume_bucket"] = pd.cut(merged["search_volume"], bins=vol_bins, labels=vol_labels)

vol_table = merged.groupby("volume_bucket").agg(
    n=("gsc_clicks", "count"),
    avg_clicks=("gsc_clicks", "mean"),
    avg_position=("gsc_avg_position", "mean")
)
display(vol_table)

/tmp/ipykernel_1829/1302740528.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  vol_table = merged.groupby("volume_bucket").agg(


,n,avg_clicks,avg_position
volume_bucket,,,
low,3821,0.059147,30.583014
medium,2906,0.088094,33.031963
high,1652,0.099879,33.856795
very_high,1779,0.064081,41.301618


**Signal 1 (CTR vs Position):** CONFIRMED — avg CTR drops steadily from 3.44% (position 1-3) to 0.30% (position 21+), across n=399 to n=10,400 rows.
**Signal 2 (Volume vs opportunity):** MIXED — avg_position worsens steadily with volume (30.6 → 41.3 across buckets), supporting the opportunity idea, but avg_clicks doesn't decrease monotonically (0.059 → 0.088 → 0.10 → 0.064) — the "high volume, low clicks" pairing only holds cleanly at the very_high tier, not across all buckets.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [83]:
import pandas as pd
import numpy as np
import os

content = pd.DataFrame(list(tables["content"]))   # full dim_content table (~519k rows)
performance = pd.DataFrame(list(tables["performance"].take(40000)))

df = performance.merge(content, on=["client_hash_id", "content_hash_id"], how="inner")
print("performance rows:", performance.shape[0])
print("content rows:", content.shape[0])
print("merged rows (before gsc filter):", df.shape[0])

# CHANGE 1: filter rows where GSC data is actually available
df = df[df["gsc_data_available"] == True].copy()

# CHANGE 2: normalize before combining (avoid scale domination)
def pct_rank(s):
    return s.rank(pct=True)

df["volume_score"] = pct_rank(df["search_volume"])
df["low_click_score"] = 1 - pct_rank(df["gsc_clicks"])
df["position_score"] = pct_rank(df["gsc_avg_position"])   # higher position number = worse rank = higher score

df["score"] = (
    0.4 * df["volume_score"]
    + 0.3 * df["low_click_score"]
    + 0.3 * df["position_score"]
)

# CHANGE 3: ONE combined reason code, not multiple comma-joined
def get_reason(row):
    if row["search_volume"] >= 30 and row["gsc_clicks"] <= 5:
        return "high_volume_low_clicks"
    if row["gsc_avg_position"] > 10:
        return "poor_ranking"
    return "general_review"

df["reason_code"] = df.apply(get_reason, axis=1)
df["action"] = "OPTIMIZE"

ranked = df.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV Saved!")
display(ranked[["content_hash_id","search_volume","gsc_clicks","gsc_avg_position","score","reason_code","action"]].head(20))

performance rows: 40000
content rows: 519606
merged rows (before gsc filter): 40000
CSV Saved!


,content_hash_id,search_volume,gsc_clicks,gsc_avg_position,score,reason_code,action
30828,content_cd12144ffdf0275e,22200.0,0,98.000000,0.860670,high_volume_low_clicks,OPTIMIZE
36142,content_db72c1748a6ba86f,4400.0,0,97.800000,0.858300,high_volume_low_clicks,OPTIMIZE
17836,content_f54e97191fc38457,6600.0,0,92.000000,0.857347,high_volume_low_clicks,OPTIMIZE
31852,content_db72c1748a6ba86f,4400.0,0,93.833333,0.857257,high_volume_low_clicks,OPTIMIZE
12695,content_4cebd80bfa17d7ff,5400.0,0,92.250000,0.857213,high_volume_low_clicks,OPTIMIZE
1273,content_62bdd508619f082f,4400.0,0,93.041667,0.857122,high_volume_low_clicks,OPTIMIZE
23156,content_62bdd508619f082f,4400.0,0,92.972973,0.856845,high_volume_low_clicks,OPTIMIZE
22213,content_62bdd508619f082f,4400.0,0,92.839080,0.856830,high_volume_low_clicks,OPTIMIZE
21508,content_62bdd508619f082f,4400.0,0,92.140351,0.856695,high_volume_low_clicks,OPTIMIZE
36276,content_358f794be8b18cab,12100.0,0,87.466667,0.856291,high_volume_low_clicks,OPTIMIZE


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [84]:
ranked_unique = ranked.sort_values("score", ascending=False).drop_duplicates(subset=["content_hash_id"], keep="first")

top20 = ranked_unique.head(20)   # 10 → 20

for i, row in top20.iterrows():
    print(f"Content: {row['content_hash_id']}")
    print(f"  Action: {row['action']}")
    print(f"  Reason: {row['reason_code']} (score={row['score']:.3f})")
    print(f"  Confidence: based on {row['reason_code']} — moderate/high")
    print(f"  What would make it wrong: if this content already has an active refresh in progress, or the volume/position numbers are from a client with very short data history")
    print()

Content: content_cd12144ffdf0275e
  Action: OPTIMIZE
  Reason: high_volume_low_clicks (score=0.861)
  Confidence: based on high_volume_low_clicks — moderate/high
  What would make it wrong: if this content already has an active refresh in progress, or the volume/position numbers are from a client with very short data history

Content: content_db72c1748a6ba86f
  Action: OPTIMIZE
  Reason: high_volume_low_clicks (score=0.858)
  Confidence: based on high_volume_low_clicks — moderate/high
  What would make it wrong: if this content already has an active refresh in progress, or the volume/position numbers are from a client with very short data history

Content: content_f54e97191fc38457
  Action: OPTIMIZE
  Reason: high_volume_low_clicks (score=0.857)
  Confidence: based on high_volume_low_clicks — moderate/high
  What would make it wrong: if this content already has an active refresh in progress, or the volume/position numbers are from a client with very short data history

Content: content

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [85]:
# Weak picks — observation
print("Observation: all 20 top picks share the same reason_code (high_volume_low_clicks).")
print("This means the rule is currently dominated by the volume_score weight (0.4) —")
print("no picks are being surfaced for poor_ranking alone or general_review.")
print("A weak pick: any row where gsc_clicks is already non-zero but volume is huge —")
print("worth checking if clicks=0 rows are new content (no time to earn clicks yet) vs genuinely underperforming.\n")

weak = top20[top20["reason_code"] == "high_volume_low_clicks"]
print(f"Rows tagged high_volume_low_clicks in top 20: {len(weak)} out of {len(top20)}")

# Leakage check — confirm no future-window or label-derived columns used
used_columns = ["search_volume", "gsc_clicks", "gsc_avg_position", "gsc_data_available"]
print("\nColumns used in scoring:", used_columns)
print("Confirmed: no is_declining_label, no trend_direction/trend_pct used (those don't exist in this warehouse table).")
print("Confirmed: no future-window columns used — gsc_clicks/gsc_avg_position/search_volume are same-row observed values, not forward-looking aggregates.")

Observation: all 20 top picks share the same reason_code (high_volume_low_clicks).
This means the rule is currently dominated by the volume_score weight (0.4) —
no picks are being surfaced for poor_ranking alone or general_review.
A weak pick: any row where gsc_clicks is already non-zero but volume is huge —
worth checking if clicks=0 rows are new content (no time to earn clicks yet) vs genuinely underperforming.

Rows tagged high_volume_low_clicks in top 20: 20 out of 20

Columns used in scoring: ['search_volume', 'gsc_clicks', 'gsc_avg_position', 'gsc_data_available']
Confirmed: no is_declining_label, no trend_direction/trend_pct used (those don't exist in this warehouse table).
Confirmed: no future-window columns used — gsc_clicks/gsc_avg_position/search_volume are same-row observed values, not forward-looking aggregates.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.